# NeceCity - train the detector on Google Colab
Run the cells from top to bottom. Before you start: **Runtime > Change runtime type > T4 GPU**.

**Files you can upload in step 2** (zip files; the names matter):
- `NeceCity_Train_Photos.zip`: your photos and their `.txt` label files (required)
- any other zip with photos + `.txt` labels, for example data made with `import_public_dataset.py` (optional)
- a zip whose name contains **negatives**: photos with NONE of the six problems (clean roads, trees, plants...). No `.txt` files needed (optional, but it is the best cure for false alarms)

At the end the notebook measures the model, suggests the confidence value to use in the app, exports `nececity_yolo.onnx` and downloads it.

## 1. Install

In [ ]:
!pip install -q ultralytics==8.3.28 onnx onnxslim onnxruntime

## 2. Upload the zip files

In [ ]:
from google.colab import files
uploaded = files.upload()          # select ALL your zip files at once
print(list(uploaded.keys()))

In [ ]:
import os, zipfile, shutil
for folder in ("photos", "negatives"):
    shutil.rmtree(folder, ignore_errors=True)
for name in uploaded:
    target = "negatives" if "negative" in name.lower() else "photos"
    with zipfile.ZipFile(name) as z:
        z.extractall(target)
    print(f"{name}  ->  {target}/")

## 3. Prepare the dataset (checks the labels, splits train / validation, counts every class)

In [ ]:
%%writefile prepare_dataset.py
"""Turns a flat folder of photos + label files into the folder layout YOLO needs.

Input folder (what you have):          Output folder (what YOLO needs):
    P1.jpg  P1.txt                         images/train/...   labels/train/...
    G3.jpg  G3.txt                         images/val/...     labels/val/...
    ...

Each .txt file has one line per object:   class_number x_centre y_centre width height
(all five numbers after the class are fractions from 0 to 1 of the photo size).

Run from the ml folder:
    python prepare_dataset.py --source ../photos --output data/real_dataset
    python prepare_dataset.py --source ../photos --negatives ../negatives --output data/real_dataset

NEGATIVE photos are photos that contain NONE of the six problems (a clean road, trees, plants, a wall...).
They teach the model what NOT to flag, which is the best cure for false alarms.
Put them in the --negatives folder (photos only, no .txt files needed). They get an empty label file.
A photo that has an EMPTY .txt file in the source folder is treated as a negative too.

What it does:
  1. checks every label file (class number 0-5, five numbers, values between 0 and 1)
  2. splits the photos into 80% training and 20% validation
     - the split is "stratified": every problem type (and the negatives) is represented in validation
     - the random seed is fixed (42), so you get the same split every time
  3. prints how many photos and boxes each class has, and warns about classes with too few photos
"""
import argparse
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

CLASS_NAMES = ["pothole", "road_damage", "garbage", "waterlogging", "damaged_road_sign", "open_drain"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
VALIDATION_SHARE = 0.20
SEED = 42
MIN_PHOTOS_PER_CLASS = 100   # below this the model usually gives many false alarms or misses


def read_labels(label_file: Path) -> list[list[float]]:
    """Reads one label file and stops with a clear message if something is wrong."""
    rows = []
    for number, line in enumerate(label_file.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise SystemExit(f"{label_file.name} line {number}: expected 5 numbers, found {len(parts)}")
        class_id = int(float(parts[0]))
        values = [float(v) for v in parts[1:]]
        if not 0 <= class_id < len(CLASS_NAMES):
            raise SystemExit(f"{label_file.name} line {number}: class {class_id} is not between 0 and {len(CLASS_NAMES) - 1}")
        if any(v < 0 or v > 1 for v in values):
            raise SystemExit(f"{label_file.name} line {number}: box values must be between 0 and 1")
        rows.append([class_id] + values)
    return rows


def list_photos(folder: Path) -> list[Path]:
    return sorted(p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)


def main() -> None:
    parser = argparse.ArgumentParser(description="Split labelled photos into train and val folders.")
    parser.add_argument("--source", required=True, help="folder with the photos and the .txt label files")
    parser.add_argument("--negatives", help="optional folder of photos that contain none of the problems")
    parser.add_argument("--output", default="data/real_dataset", help="folder to create")
    args = parser.parse_args()

    source, output = Path(args.source), Path(args.output)
    photos = list_photos(source)
    if not photos:
        raise SystemExit(f"No photos found in {source}")

    # photo path -> list of label rows ([] for a negative photo)
    labels: dict[Path, list[list[float]]] = {}
    for photo in photos:
        label_file = photo.with_suffix(".txt")
        if not label_file.exists():
            raise SystemExit(f"Missing label file for {photo.name} (expected {label_file.name}). "
                             f"If this photo has no problem, put it in the --negatives folder instead.")
        labels[photo] = read_labels(label_file)   # an empty label file means "no problem here" (a negative photo)

    negatives = list_photos(Path(args.negatives)) if args.negatives else []
    for photo in negatives:
        labels[photo] = []

    # file names must be unique, because all photos end up in the same folder
    names = Counter(p.name for p in labels)
    clashes = [n for n, c in names.items() if c > 1]
    if clashes:
        raise SystemExit(f"These file names are used more than once: {clashes[:5]}. Rename them so every name is unique.")

    # group photos by their most common class (negatives form their own group), then pick validation photos
    groups = defaultdict(list)
    for photo, rows in labels.items():
        key = Counter(int(r[0]) for r in rows).most_common(1)[0][0] if rows else -1
        groups[key].append(photo)

    rng = random.Random(SEED)
    validation = set()
    for key, members in sorted(groups.items()):
        members.sort()
        rng.shuffle(members)
        validation.update(members[:max(1, round(len(members) * VALIDATION_SHARE))])

    if output.exists():
        shutil.rmtree(output)
    for photo, rows in labels.items():
        part = "val" if photo in validation else "train"
        (output / "images" / part).mkdir(parents=True, exist_ok=True)
        (output / "labels" / part).mkdir(parents=True, exist_ok=True)
        shutil.copy2(photo, output / "images" / part / photo.name)
        text = "\n".join(" ".join(str(v) for v in row) for row in rows)
        if rows:
            shutil.copy2(photo.with_suffix(".txt"), output / "labels" / part / (photo.stem + ".txt"))
        else:
            (output / "labels" / part / (photo.stem + ".txt")).write_text("")   # empty file = background photo

    # summary
    total = len(labels)
    print(f"Photos: {total}  (train {total - len(validation)}, val {len(validation)}),  of which negatives: {len(negatives)}")
    print(f"{'class':20s} {'photos':>7s} {'boxes':>7s} {'boxes in val':>13s}")
    for class_id, name in enumerate(CLASS_NAMES):
        photo_count = sum(any(int(r[0]) == class_id for r in rows) for rows in labels.values())
        box_count = sum(int(r[0]) == class_id for rows in labels.values() for r in rows)
        val_boxes = sum(int(r[0]) == class_id for p in validation for r in labels[p])
        warning = "   <-- too few photos" if photo_count < MIN_PHOTOS_PER_CLASS else ""
        print(f"{name:20s} {photo_count:7d} {box_count:7d} {val_boxes:13d}{warning}")
    print(f"Done. Dataset written to {output}")
    if any(sum(any(int(r[0]) == c for r in rows) for rows in labels.values()) < MIN_PHOTOS_PER_CLASS for c in range(len(CLASS_NAMES))):
        print(f"\nTip: aim for at least {MIN_PHOTOS_PER_CLASS} photos per class. See ml/README.md for where to get more.")


if __name__ == "__main__":
    main()

In [ ]:
import os
extra = "--negatives negatives" if os.path.isdir("negatives") else ""
!python prepare_dataset.py --source photos {extra} --output data/real_dataset

In [ ]:
yaml_text = '''path: /content/data/real_dataset
train: images/train
val: images/val
names:
  0: pothole
  1: road_damage
  2: garbage
  3: waterlogging
  4: damaged_road_sign
  5: open_drain
'''
open('dataset.yaml', 'w').write(yaml_text)

## 4. Train
Choose the model size: **`yolov8n.pt`** is small and fast (good for under about 300 photos); **`yolov8s.pt`** is bigger and usually more accurate when you have 300+ photos. Both can be exported to ONNX and run on the free server (`yolov8s` is slower on its tiny CPU).

In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolov8n.pt"      # or "yolov8s.pt"
EPOCHS = 150                   # use 200 when you have many photos

model = YOLO(MODEL_NAME)       # COCO-pretrained starting point (transfer learning)
model.train(
    data="dataset.yaml", epochs=EPOCHS, imgsz=640, batch=16, patience=40, seed=42,
    degrees=10, fliplr=0.5, hsv_s=0.7, hsv_v=0.4, scale=0.5, mosaic=1.0,
    project="runs", name="nececity_yolo",
)
best_path = str(model.trainer.best)
print("Best model:", best_path)

## 4b. Check that the training really worked
Ultralytics keeps the checkpoint with the best score on the validation photos as `best.pt`. With very few validation photos that score is noisy, so "best" can be an **early, half-trained epoch**. This cell shows which epoch was best. If it was early, the notebook uses `last.pt` (the final epoch) instead.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{model.trainer.save_dir}/results.csv")
df.columns = [c.strip() for c in df.columns]
fitness = 0.1 * df["metrics/mAP50(B)"] + 0.9 * df["metrics/mAP50-95(B)"]
best_epoch = int(fitness.idxmax()) + 1
epochs_run = len(df)
print(f"Epochs run: {epochs_run}   best epoch according to Ultralytics: {best_epoch}")
print(df[["epoch", "train/box_loss", "train/cls_loss", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)"]].tail(5).round(3).to_string(index=False))

USE_LAST = best_epoch < 0.6 * epochs_run     # best epoch is early: the model was still learning
final_path = str(model.trainer.last) if USE_LAST else best_path
print("\nUsing:", "last.pt (final epoch)" if USE_LAST else "best.pt", "->", final_path)

## 5. Measure the result on the validation photos

In [ ]:
best = YOLO(final_path)
results = best.val(data="dataset.yaml", imgsz=640)
print("Precision    :", round(float(results.box.mp), 3), " (of the boxes it drew, how many were right)")
print("Recall       :", round(float(results.box.mr), 3), " (of the real problems, how many it found)")
print("mAP@0.5      :", round(float(results.box.map50), 3))
print()
for class_id, score in zip(results.box.ap_class_index, results.box.ap50):
    print(f"{best.names[int(class_id)]:20s} mAP@0.5 = {float(score):.3f}")

## 5b. Can the model recognise its OWN training photos?
A trained model should find the labelled problems in almost all of the photos it was trained on (about 90% or more). If it does not, something is wrong with the labels or the training, and more photos will not fix that: run `visualize_labels.py` on your photos folder and look at the boxes.

In [ ]:
import glob, os

def labelled_classes(label_path):
    if not os.path.exists(label_path):
        return set()
    return {int(float(l.split()[0])) for l in open(label_path).read().splitlines() if l.strip()}

found = total = 0
problems = []
for img in sorted(glob.glob("data/real_dataset/images/train/*")):
    want = labelled_classes(img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt")
    if not want:
        continue                               # negative photo
    got = {int(b.cls) for b in best.predict(source=img, conf=0.25, verbose=False)[0].boxes}
    total += 1
    if want <= got:
        found += 1
    else:
        problems.append((os.path.basename(img), [best.names[c] for c in want], [best.names[c] for c in got]))
print(f"Training photos where every labelled class was found: {found} of {total}")
for name, want, got in problems[:15]:
    print(f"  {name:24s} labelled {want}   model says {got}")

## 6. Choose the confidence value for the app
A model draws boxes with a confidence from 0 to 1. A LOW limit finds more problems but gives many false alarms; a HIGH limit gives few false alarms but misses real problems. This table tests several limits. Pick the row with the best **F1** (a balance of both). Use that value as `MODEL_MIN_CONFIDENCE` in the app.

In [ ]:
rows = []
for c in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]:
    m = best.val(data="dataset.yaml", imgsz=640, conf=c, verbose=False, plots=False)
    p, r = float(m.box.mp), float(m.box.mr)
    rows.append((c, p, r, 2 * p * r / (p + r) if (p + r) else 0.0))
print(f"{'conf':>5s} {'precision':>10s} {'recall':>8s} {'F1':>6s}")
for c, p, r, f1 in rows:
    print(f"{c:5.2f} {p:10.3f} {r:8.3f} {f1:6.3f}")
best_conf = max(rows, key=lambda x: x[3])[0]
print("\nSuggested MODEL_MIN_CONFIDENCE =", best_conf)

m = best.val(data="dataset.yaml", imgsz=640, conf=best_conf, verbose=False, plots=False)
print("\nPer class at that confidence (a class with LOW precision gives false alarms: give it a higher limit in the app):")
for class_id, p, r in zip(m.box.ap_class_index, m.box.p, m.box.r):
    print(f"{best.names[int(class_id)]:20s} precision {float(p):.2f}   recall {float(r):.2f}")

## 7. Look at the predictions on the validation photos (using the suggested confidence)

In [ ]:
from PIL import Image
from IPython.display import display

for r in best.predict(source="data/real_dataset/images/val", conf=best_conf, verbose=False):
    display(Image.fromarray(r.plot()[..., ::-1]))

## 8. (Optional) Compare with an older model
If an older model seemed to work better, upload its `.pt` file here. For every validation photo this prints how many boxes each model draws; the better model draws about the right number, not far too many.

In [ ]:
old_upload = files.upload()        # choose the OLD .pt file, or skip this cell
old_name = list(old_upload.keys())[0]
old = YOLO(old_name)
import glob
print(f"{'photo':25s} {'new model':>10s} {'old model':>10s}")
for path in sorted(glob.glob("data/real_dataset/images/val/*")):
    n_new = len(best.predict(source=path, conf=best_conf, verbose=False)[0].boxes)
    n_old = len(old.predict(source=path, conf=best_conf, verbose=False)[0].boxes)
    print(f"{os.path.basename(path):25s} {n_new:10d} {n_old:10d}")

## 9. Export to ONNX (the file the web app uses) and check it

In [ ]:
import numpy as np, onnxruntime as ort

onnx_path = best.export(format="onnx", imgsz=640, opset=12)
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
output = session.run(None, {session.get_inputs()[0].name: np.zeros((1, 3, 640, 640), np.float32)})[0]
print("ONNX output shape:", output.shape, "(expected (1, 10, 8400))")

## 10. Download the two files
Put both in the `ml/weights` folder of your project, commit the `.onnx`, and set the confidence value suggested in step 6.

In [ ]:
shutil.copy(final_path, "nececity_yolo.pt")
shutil.copy(onnx_path, "nececity_yolo.onnx")
files.download("nececity_yolo.onnx")
files.download("nececity_yolo.pt")